# Phase 2: preprocessing and reproducible splits
Prepare a class-covered development sample, clean features, prevent exact duplicate leakage, and fit reusable preprocessing. **No classifier training.** Select the **NetworkMonitoring (Python 3.12)** kernel. Install `requirements.txt` in that environment if imports fail.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Start Jupyter from the project root or notebooks directory")
sys.path.insert(0, str(ROOT))
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from src.prepare_data import prepare
from src.preprocessing import PROVENANCE, build_preprocessor
print("Python:", sys.executable)

## 1. Configuration and sampling
The first run scans every source file (about 885 MB); later runs reuse the cached sample. Random priorities sample across each entire class, avoiding early-file bias. A per-class cap changes prevalence: this is a development benchmark. Sampling and deduplication do not guarantee independent sessions or hosts.

In [ ]:
PER_CLASS = 5000
SEED = 42
REVISION = "b33f3052398ffd63463ce3120fec2acbb68e2ff6"
summary = prepare(per_class=PER_CLASS, seed=SEED, revision=REVISION)
folder = ROOT / "data/processed/phase2"

## 2. Cleaning decisions and retained records
Observed web-attack separators are normalized. Invalid numeric strings, infinities, and -1 initial-window sentinels become missing. Other negatives remain. Conflicting-label feature groups are excluded; exact duplicates are removed before splitting. Tiny classes are flagged, not silently discarded.

In [ ]:
display(pd.Series(summary["cleaning"], name="records").to_frame())
print("Training-only classes:", summary["train_only_classes"])
print("Holdout classes absent:", summary["missing_holdout_classes"])
train_raw = pd.read_parquet(folder / "train_raw.parquet")
display(train_raw.head())

## 3. Class coverage and split sizes
Approximately 70/15/15 per class, with at least one validation/test record when there are at least three unique rows. Overlap purging can subsequently reduce holdouts. Tiny holdouts cannot support reliable per-class metrics.

In [ ]:
counts = pd.DataFrame({name: item["classes"] for name, item in summary["splits"].items()}).fillna(0).astype(int)
display(counts)
counts.plot.barh(figsize=(11, 7), logx=True)
plt.xlabel("Records (log scale)")
plt.title("Phase 2 development sample: class coverage")
plt.tight_layout()
plt.show()

## 4. Reusable preprocessing
The pipeline fits median imputation, constant removal, and scaling **only on training data**. Validation and test use transform only. For future cross-validation, build a new pipeline inside every fold; do not reuse fitted transforms across folds.

In [ ]:
artifact = joblib.load(ROOT / "models/phase2_preprocessor.joblib")
pipeline = artifact["preprocessor"]
display(pipeline)
print("Retained features:", len(summary["feature_names"]))
print("Dropped training constants:", summary["removed_constant_features"])
display(pd.Series(summary["label_mapping"], name="multiclass_id").to_frame())
print("Binary mapping:", summary["binary_mapping"])

## 5. Validate saved artifacts
Source IDs and transformed feature rows must be disjoint across splits. Label and provenance columns are excluded from features. Persistence checks confirm the saved pipeline reproduces the saved test matrix; no model scores are computed.

In [ ]:
arrays = {}
raw = {}
for name in ("train", "validation", "test"):
    with np.load(folder / f"{name}.npz") as data:
        arrays[name] = data["X"].copy()
        assert np.isfinite(arrays[name]).all()
        assert len(data["y_binary"]) == len(arrays[name])
    raw[name] = pd.read_parquet(folder / f"{name}_raw.parquet")
    assert len(raw[name]) == len(arrays[name])
    print(name, arrays[name].shape)
assert not pd.concat([part[PROVENANCE] for part in raw.values()]).duplicated().any()
for a, b in (("train", "validation"), ("train", "test"), ("validation", "test")):
    assert not pd.MultiIndex.from_frame(pd.DataFrame(arrays[a])).isin(pd.MultiIndex.from_frame(pd.DataFrame(arrays[b]))).any()
np.testing.assert_allclose(pipeline.transform(raw["test"][artifact["raw_feature_columns"]]), arrays["test"])
print("Finite values, disjoint splits, and saved pipeline checks passed.")

## 6. Preparation report and next phase
Review class support, altered prevalence, and residual temporal/session similarity before training. Phase 3 will compare classifiers; it has not started.

In [ ]:
display(Markdown((ROOT / "reports/phase2_preprocessing_summary.md").read_text()))